# PA-02 · Declara el contrato de ingesta antes de fiarte de la tabla

**AFIRMACIÓN → COMPROBACIÓN → PASS / FAIL / UNKNOWN**

Clase 2 · Unidad 2 — Ingesta, formatos, esquemas y fundamentos de pandas
Datos: *International Airline Activity* (BITRE), tabla 1, de enero de 2009 a noviembre de 2025

| | |
|---|---|
| **Núcleo de clase** | ≈ 28 min entre trabajo y puesta en común (partes 0 a 5). Reparto orientativo: lectura inicial y parte 0: 2,5 · parte 1: 3,5 · parte 2: 3,5 · parte 3: 6,5 · parte 4: 3,5 · parte 5: 3,5 · puesta en común: 5 |
| **Para casa** | ≤ 20 min (parte 6) |
| **Qué entregas** | Este mismo notebook, con tu contrato completado |
| **Competencia** | ENTENDER → DECIDIR → IMPLEMENTAR → VALIDAR → DEFENDER |

## El problema

Un compañero te pasa una tabla de actividad aérea internacional y te dice:

> «Los datos están limpios. Cárgalos y sigue.»

Cargar el fichero es una línea de código. Lo difícil es otra cosa: **¿qué puedes afirmar de esta tabla, y con qué evidencia?**

En esta actividad no limpias nada ni haces ningún análisis. Redactas un **contrato de ingesta**: afirmaciones sobre la tabla que se pueden **comprobar**. Para cada una decides **PASS**, **FAIL** o **UNKNOWN**.

## Las tres respuestas posibles

| Veredicto | Significa |
|---|---|
| **PASS** | La evidencia disponible respalda la afirmación. |
| **FAIL** | La evidencia disponible la contradice. |
| **UNKNOWN** | **No es verificable con la evidencia de la que disponemos ahora.** |

UNKNOWN no es «no sé» ni un FAIL disimulado. Es una afirmación precisa sobre la evidencia: dices **qué falta** y **qué la resolvería**. Y un FAIL tampoco es un fracaso tuyo: es un hallazgo.

### Qué evidencia tienes (y cuál no)

Trabajas con una **evidencia deliberadamente limitada**: la tabla, la hoja ReadMe del origen, la leyenda de símbolos y una tabla de totales publicados. Esa es la condición de la actividad. El origen publica más documentación, que no forma parte de este paquete.

Para cada afirmación distingue siempre tres cosas:

1. **Qué demuestran los datos**: lo que puedes comprobar ejecutando código sobre la tabla.
2. **Qué demuestra la documentación**: lo que dicen el ReadMe y la leyenda.
3. **Qué queda sin demostrar.**

### Evidencia proporcionada y evidencia externa

- **Evidencia proporcionada**: lo que hay en este paquete (tabla, ReadMe, leyenda, totales) y lo que tú compruebas ejecutando código sobre ello.
- **Evidencia externa**: cualquier otra cosa: documentación del origen que encuentres, conocimiento propio, una búsqueda, una IA.

Reglas:

1. Los veredictos se deciden con la **evidencia proporcionada**. Si no basta para establecer una afirmación, el veredicto es **UNKNOWN**.
2. **No marques FAIL** porque te parezca poco plausible, porque los datos sugieran una posible explicación, porque lo recuerdes o porque una IA lo afirme.
3. Puedes usar evidencia externa y **no se penaliza por ser externa**, pero tienes que **identificarla** (fuente) en el campo `evidencia_externa` y dejar claro **qué establece la evidencia proporcionada y qué añade la externa**. Un veredicto distinto de UNKNOWN apoyado en evidencia externa solo es válido si la identificas y reconoces que tu conclusión depende de ella.

## Qué te damos

| Fichero (carpeta `../data/`) | Qué es |
|---|---|
| `international_airline_activity_2009-01_2025-11.csv` | La tabla (hoja *Data* del libro de origen exportada a CSV; valores sin modificar) |
| `docs/ReadMe_sheet.md` | Hoja *ReadMe* del origen, copia literal |
| `docs/symbols_legend.md` | Leyenda de símbolos del origen, copia literal |
| `totals_published.csv` | Totales publicados por año del origen (para una reconciliación) |
| `docs/DATASET_NOTICE.md` | Procedencia, atribución y qué se ha adaptado |

## Cómo se trabaja: DECLARA, LUEGO VERIFICA

1. **Declaras** lo que esperas *antes* de ejecutar ninguna comprobación.
2. **Verificas** con una comprobación ejecutable.
3. Si el resultado te sorprende, esa sorpresa es lo valioso: apunta por qué.

## Sobre el uso de IA

Puedes usar documentación externa y herramientas de IA para generar código o proponer hipótesis. **La IA puede generar; tú sigues siendo responsable** de entenderlo, comprobarlo, ponerlo a prueba y defenderlo. Una respuesta generada que no has comprobado sobre esta tabla no cuenta como evidencia proporcionada: si la usas, identifícala como evidencia externa (ver arriba).

---
## Parte 0 · Preparación

Ejecuta la celda. No hay nada que decidir aquí.

In [ ]:
# SOLO PARA GOOGLE COLAB (añadido para el repositorio; en Jupyter o VS Code esta celda no hace nada).
# Descarga desde GitHub la carpeta data/ de esta práctica y se sitúa en student/,
# para que las rutas "../data/..." del notebook funcionen igual que en local.
import os, sys
if "google.colab" in sys.modules:
    import urllib.parse, urllib.request
    BASE = "https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-02/"
    FICHEROS = [
        'data/docs/ReadMe_sheet.md',
        'data/docs/symbols_legend.md',
        'data/international_airline_activity_2009-01_2025-11.csv',
        'data/totals_published.csv',
    ]
    for f in FICHEROS:
        destino = os.path.join("/content/PA-02", f)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        if not os.path.exists(destino):
            urllib.request.urlretrieve(BASE + urllib.parse.quote(f), destino)
    os.makedirs("/content/PA-02/student", exist_ok=True)
    os.chdir("/content/PA-02/student")
    print(f"Colab: {len(FICHEROS)} ficheros de datos listos en /content/PA-02/data")

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

RUTA_CSV = "../data/international_airline_activity_2009-01_2025-11.csv"
RUTA_TOTALES = "../data/totals_published.csv"
RUTA_README = "../data/docs/ReadMe_sheet.md"
RUTA_LEYENDA = "../data/docs/symbols_legend.md"

medidas = ["Passengers In", "Freight In", "Mail In", "Passengers Out", "Freight Out", "Mail Out"]


def comprobar(condicion, texto):
    """Como un assert, pero no detiene el notebook: imprime ✔ si se cumple y ✘ si no."""
    print(("✔ " if condicion else "✘ ") + texto)


print("pandas", pd.__version__)
if pd.__version__ != "3.0.6":
    print("Aviso: el curso usa pandas 3.0.6; con otra versión algunos tipos y resultados pueden diferir.")

---
## Parte 1 · ENTENDER: qué tenemos (≈ 3,5 min)

### 1.1 La documentación que se te ha dado
Léela con calma: es toda la documentación de la que dispones.

In [ ]:
display(Markdown(open(RUTA_README, encoding="utf-8").read()))
display(Markdown(open(RUTA_LEYENDA, encoding="utf-8").read()))

### 1.2 El fichero, antes de pandas

Mira el texto tal cual está en el disco. Primero las cuatro primeras líneas y después tres líneas que contienen `..`.

In [ ]:
with open(RUTA_CSV, encoding="utf-8") as f:
    lineas = f.read().splitlines()

print("Líneas del fichero (incluida la cabecera):", len(lineas))
print()
for linea in lineas[:4]:
    print(linea)
print("...")
for linea in [l for l in lineas if ".." in l][:3]:
    print(linea)

### 1.3 La carga que haría tu compañero

In [ ]:
df_crudo = pd.read_csv(RUTA_CSV)
print(df_crudo.shape)
df_crudo.head()

> **Pregunta silenciosa.** La carga no ha dado ningún error. ¿Qué has comprobado realmente con eso? Todavía no sigas explorando la tabla: primero declara.

### 1.4 ¿Qué es una fila?

Según el ReadMe y las primeras filas, completa dos cosas: qué representa una fila y qué columnas, juntas, la identifican.

In [ ]:
mi_grano = ""   # Completa: «Una fila representa ...»
mi_clave = []   # Completa: lista con los nombres de las columnas que, juntas, identifican una fila

---
## Parte 2 · DECIDIR (declarar): tu contrato de expectativas (≈ 3,5 min)

Este es el menú de afirmaciones candidatas sobre la tabla. Cada una puede resultar cierta, falsa o imposible de decidir con lo que tenemos.

| Bloque | Id | Afirmación |
|---|---|---|
| **A** · Estructura y tipos | **C1** | Todas las columnas del CSV están descritas en el ReadMe. |
|  | **C2** | Las seis columnas de medidas (`Passengers`, `Freight` y `Mail`, en «In» y «Out») se cargan como numéricas. |
| **B** · Marcadores de ausencia y de cero | **C3** | Los valores ausentes llegan como celdas vacías y se cargan como NaN. |
|  | **C4** | El cero se escribe con el símbolo `-`, como dice la leyenda de símbolos. |
| **C** · Fecha y grano | **C5** | `Month` se carga como fecha y no falta ningún mes entre el primero y el último. |
|  | **C6** | `Year` es el año natural (de calendario) de `Month`. |
|  | **C7** | Hay una sola fila por cada combinación de `Month`, `Scheduled Operator` y `Country to/from`. |
| **D** · Valores y significado | **C8** | Ninguna medida es negativa y los pasajeros son números enteros. |
|  | **C9** | ★ Cada cadena de `Scheduled Operator` identifica a una aerolínea, y cada aerolínea aparece siempre con la misma cadena durante todo el periodo. |
|  | **C10** | Los totales publicados año a año (`totals_published.csv`) coinciden con la suma de la tabla de datos. |
|  | **C11** | Un `0` en `Mail In` / `Mail Out` significa «ese operador no transportó correo ese mes». |
|  | **C12** | Un `..` en las columnas de pasajeros significa que ese servicio fue solo de carga (no hubo servicio de pasajeros). |

**Elige entre 4 y 6** (o reescribe alguna con tus palabras: usa su mismo identificador y añade `-R`). Tu elección debe cubrir **al menos 3 de los 4 bloques** y **al menos una afirmación del bloque D**. La afirmación **C9** (★) lleva más tiempo: si la eliges, es mejor dejarla para casa.

**Todavía no ejecutes ninguna comprobación.** Para cada afirmación elegida, escribe tu **predicción** (`"PASS"`, `"FAIL"` o `"UNKNOWN"`) y **por qué** lo esperas en una frase.

In [ ]:
MENU = {
    "C1": "Todas las columnas del CSV están descritas en el ReadMe.",
    "C2": "Las seis columnas de medidas (`Passengers`, `Freight` y `Mail`, en «In» y «Out») se cargan como numéricas.",
    "C3": "Los valores ausentes llegan como celdas vacías y se cargan como NaN.",
    "C4": "El cero se escribe con el símbolo `-`, como dice la leyenda de símbolos.",
    "C5": "`Month` se carga como fecha y no falta ningún mes entre el primero y el último.",
    "C6": "`Year` es el año natural (de calendario) de `Month`.",
    "C7": "Hay una sola fila por cada combinación de `Month`, `Scheduled Operator` y `Country to/from`.",
    "C8": "Ninguna medida es negativa y los pasajeros son números enteros.",
    "C9": "★ Cada cadena de `Scheduled Operator` identifica a una aerolínea, y cada aerolínea aparece siempre con la misma cadena durante todo el periodo.",
    "C10": "Los totales publicados año a año (`totals_published.csv`) coinciden con la suma de la tabla de datos.",
    "C11": "Un `0` en `Mail In` / `Mail Out` significa «ese operador no transportó correo ese mes».",
    "C12": "Un `..` en las columnas de pasajeros significa que ese servicio fue solo de carga (no hubo servicio de pasajeros).",
}
BLOQUES = {'A': ['C1', 'C2'], 'B': ['C3', 'C4'], 'C': ['C5', 'C6', 'C7'], 'D': ['C8', 'C9', 'C10', 'C11', 'C12']}

In [ ]:
mi_contrato = {
    # "C?": {"prediccion": "PASS | FAIL | UNKNOWN", "porque": "una frase"},
}

In [ ]:
# Revisión de la forma de tu contrato (no valora si tus predicciones son acertadas)
elegidas = [cid.split("-")[0] for cid in mi_contrato]
bloques_cubiertos = [b for b, ids in BLOQUES.items() if any(cid in ids for cid in elegidas)]
comprobar(4 <= len(mi_contrato), f"Tienes {len(mi_contrato)} afirmaciones (mínimo 4)")
comprobar(all(v.get("prediccion") in ("PASS", "FAIL", "UNKNOWN") for v in mi_contrato.values()) and len(mi_contrato) > 0,
          "Todas tienen una predicción válida (PASS, FAIL o UNKNOWN)")
comprobar(len(bloques_cubiertos) >= 3, f"Cubres {len(bloques_cubiertos)} bloques (mínimo 3): {bloques_cubiertos}")
comprobar("D" in bloques_cubiertos, "Incluyes al menos una afirmación del bloque D")

---
## Parte 3 · IMPLEMENTAR y VERIFICAR: comprobaciones (≈ 6,5 min)

Hay una comprobación escrita por afirmación. **Ejecuta solo las de tu contrato.** Cada una imprime evidencia; el veredicto lo pones tú.

> **Código proporcionado — no se evalúa esta sintaxis.** Algunas comprobaciones usan funciones que no has tenido que aprender (por ejemplo `pd.to_datetime`, `.dt`, `pd.to_numeric`, `groupby().agg`). No tienes que escribirlas ni memorizarlas. Sí eres responsable de **entender qué consigue el código, si su resultado es válido y qué significa**.

Una buena evidencia es una cifra o un hecho observable, no una opinión.

### 3.1 Preparación común (ejecútala una vez)

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
# Copias SOLO PARA COMPROBAR (no son una decisión de lectura):
num = df_crudo[medidas].apply(pd.to_numeric, errors="coerce")   # lo que no sea número pasa a NaN
fechas = pd.to_datetime(df_crudo["Month"])
totales = pd.read_csv(RUTA_TOTALES, index_col="Year")
totales.columns = medidas                                       # mismas seis medidas, en el mismo orden

# to_numeric(errors="coerce") convierte en NaN todo lo que no entiende. ¿Qué ha convertido?
n_convertidas = int(num.isna().sum().sum())
n_marcadores = int((df_crudo[medidas] == "..").sum().sum())
comprobar(n_convertidas == n_marcadores, f"La conversión solo ha afectado a las celdas con '..' ({n_convertidas} celdas)")

**C1** · ¿Hay columnas en el CSV que el ReadMe no describa, o al revés?

In [ ]:
documentadas = ["Month", "Scheduled Operator", "Country to/from", "Passengers In", "Freight In",
                "Mail In", "Passengers Out", "Freight Out", "Mail Out"]      # lista de campos del ReadMe
print("Columnas del CSV:", df_crudo.shape[1])
print("En el CSV pero no en el ReadMe:", sorted(set(df_crudo.columns) - set(documentadas)))
print("En el ReadMe pero no en el CSV:", sorted(set(documentadas) - set(df_crudo.columns)))

**C2** · Tipos de las seis medidas.

In [ ]:
print(df_crudo[medidas].dtypes)
print()
print("¿Es numérica?", {c: pd.api.types.is_numeric_dtype(df_crudo[c]) for c in medidas})

**C3** · Valores ausentes: qué dice `isna()` y qué dice `info()`.

In [ ]:
print("Celdas NaN por columna:")
print(df_crudo.isna().sum())
print()
print("Celdas con el texto '..' en las medidas:")
print((df_crudo[medidas] == "..").sum())
print()
df_crudo.info(memory_usage=False)

**C4** · El cero: ¿se escribe `-`? ¿Hay ceros? ¿Cómo están escritos?

In [ ]:
print("Celdas con el texto '-' en las medidas:", (df_crudo[medidas] == "-").sum().to_dict())
print("Celdas con el número 0 en las medidas:  ", (num == 0).sum().to_dict())

**C5** · El tipo de `Month` y los meses que hay entre el primero y el último.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
print("Tipo de Month tal como se carga:", df_crudo["Month"].dtype)
esperados = (fechas.max().year - fechas.min().year) * 12 + (fechas.max().month - fechas.min().month) + 1
print("Primer y último mes:", fechas.min().date(), "→", fechas.max().date())
print("Meses distintos:", fechas.nunique(), "| meses que habría sin huecos:", esperados)
print("Todas las fechas son día 1:", bool((fechas.dt.day == 1).all()))

**C6** · La relación entre `Year` y el año de `Month`.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
print("Filas en las que Year != año de Month:", int((df_crudo["Year"] != fechas.dt.year).sum()))
print()
print(df_crudo.assign(fecha=fechas).groupby("Year")["fecha"].agg(["min", "max", "nunique"]).head(3))

**C7** · Unicidad de la clave que declaraste en la parte 1.4 (y un contraste).

In [ ]:
if mi_clave:
    print("Filas repetidas según tu clave", mi_clave, ":", int(df_crudo.duplicated(mi_clave).sum()))
else:
    print("Declara primero tu clave en mi_clave (parte 1.4).")
print("Contraste, solo con ['Month', 'Scheduled Operator']:", int(df_crudo.duplicated(["Month", "Scheduled Operator"]).sum()))

**C8** · Valores negativos y pasajeros no enteros.

In [ ]:
print("Mínimo de cada medida:", num.min().to_dict())
print("Pasajeros con decimales:", {c: int((num[c].dropna() % 1 != 0).sum()) for c in ["Passengers In", "Passengers Out"]})

**C9 ★** · Cadenas de operador. Pares de cadenas en los que una es el comienzo de la otra, con su periodo y los meses en que aparecen (tarda más: mejor para casa).

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
vida = df_crudo.assign(fecha=fechas).groupby("Scheduled Operator")["fecha"].agg(["min", "max", "nunique"])
print("Cadenas distintas de operador:", len(vida))
nombres = sorted(vida.index)
pares = [(a, b) for a in nombres for b in nombres if a != b and b.startswith(a)]   # una cadena es el comienzo de otra
for a, b in pares:
    print(f"{a} [{vida.loc[a, 'min'].date()} → {vida.loc[a, 'max'].date()}, {vida.loc[a, 'nunique']} meses]"
          f"   |   {b} [{vida.loc[b, 'min'].date()} → {vida.loc[b, 'max'].date()}, {vida.loc[b, 'nunique']} meses]")

**C10** · Reconciliación con los totales publicados, año a año. Los decimales se comparan con tolerancia (aquí 1e-6).

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
suma_year = num.assign(Year=df_crudo["Year"]).groupby("Year")[medidas].sum()
suma_natural = num.assign(Year=fechas.dt.year).groupby("Year")[medidas].sum()
print("Diferencia máxima, agrupando por la columna Year:", (suma_year - totales).abs().max().max())
print("Diferencia máxima, agrupando por el año de Month: ", (suma_natural - totales).abs().max().max())

**C11** · Ceros en las columnas de correo.

In [ ]:
print("Proporción de filas con 0:", (num[["Mail In", "Mail Out"]] == 0).mean().round(3).to_dict())
correo = df_crudo[["Mail In", "Mail Out"]]
print("Celdas '-' en correo:", int((correo == "-").sum().sum()), "| celdas '..' en correo:", int((correo == "..").sum().sum()))

**C12** · `..` en Passengers In: quién lo tiene y qué más ocurre en esas filas.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
marca = df_crudo["Passengers In"] == ".."
print("Filas con '..' en Passengers In:", int(marca.sum()))
por_op = marca.groupby(df_crudo["Scheduled Operator"]).agg(["sum", "size"])
por_op["proporcion"] = (por_op["sum"] / por_op["size"]).round(3)
print(por_op[por_op["sum"] > 0].sort_values("sum", ascending=False).head(8))
print("Filas con '..' en Passengers In pero pasajeros > 0 en Passengers Out:", int((marca & (num["Passengers Out"] > 0)).sum()))

> **Pregunta silenciosa.** Si has ejecutado C3, habrás visto que `info()` dice «21588 non-null» en todas las columnas. ¿Significa eso que no falta ningún dato? ¿Qué has visto en tus comprobaciones que `info()` no te cuenta?

### 3.2 Tus veredictos

Para cada afirmación de tu contrato escribe el veredicto y la evidencia (una cifra o un hecho). Si es **UNKNOWN**, escribe también **qué evidencia lo resolvería**. Si has usado **evidencia externa**, identifícala en `evidencia_externa` (recuerda las reglas de arriba).

In [ ]:
veredictos = {
    # "C?": {"veredicto": "PASS | FAIL | UNKNOWN",
    #        "evidencia": "una cifra o un hecho observable",
    #        "que_lo_resolveria": "solo si es UNKNOWN: qué evidencia haría falta",
    #        "evidencia_externa": "solo si usas evidencia que NO está en el paquete: fuente + qué añade"},
}

In [ ]:
# Tu contrato: predicción frente a veredicto
filas = []
for cid, p in mi_contrato.items():
    v = veredictos.get(cid, {})
    filas.append({
        "id": cid,
        "afirmacion": MENU.get(cid.split("-")[0], "(tuya)"),
        "prediccion": p.get("prediccion", ""),
        "veredicto": v.get("veredicto", ""),
        "sorpresa": bool(v) and p.get("prediccion") != v.get("veredicto"),
        "evidencia": v.get("evidencia", ""),
        "que_lo_resolveria": v.get("que_lo_resolveria", ""),
        "evidencia_externa": v.get("evidencia_externa", ""),
    })
contrato = pd.DataFrame(filas, columns=["id", "afirmacion", "prediccion", "veredicto", "sorpresa", "evidencia", "que_lo_resolveria", "evidencia_externa"])
print("Veredictos:", contrato["veredicto"].replace("", "(pendiente)").value_counts().to_dict(), "| sorpresas:", int(contrato["sorpresa"].sum()))
contrato[["id", "prediccion", "veredicto", "sorpresa", "evidencia"]]

---
## Parte 4 · DECIDIR e IMPLEMENTAR la lectura (≈ 3,5 min)

Con la evidencia de la parte 3, decide **cómo debe leerse el fichero** para que la tabla en memoria no diga más (ni menos) que el origen.

| Opción de `pd.read_csv` | Qué hace |
|---|---|
| `na_values=[...]` | Lista de textos que se leerán como ausentes (NaN). |
| `parse_dates=[...]` | Lista de columnas que se leerán como fechas. |
| `dtype={...}` | Fija el tipo de columnas concretas. |
| `keep_default_na=False` | Desactiva la lista de textos de ausencia que pandas aplica por defecto. |

Rellena tus decisiones. La celda siguiente lee el fichero con ellas (**código proporcionado — no se evalúa esta sintaxis**, incluida la opción `parse_dates`).

In [ ]:
decisiones = {
    "marcadores_ausentes": [],   # textos que se leerán como ausentes (NaN)
    "columnas_fecha": [],        # columnas que se leerán como fecha
    "columnas_numericas": [],    # columnas que DEBEN salir numéricas
    "year_es_calendario": None,  # True o False: ¿puedo usar Year como año natural?
    "justificacion": "",         # 2-3 líneas: por qué estas decisiones
}

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
df_leido = pd.read_csv(
    RUTA_CSV,
    sep=",",
    encoding="utf-8",
    na_values=decisiones["marcadores_ausentes"],
    parse_dates=decisiones["columnas_fecha"],
)
df_leido.dtypes

---
## Parte 5 · VALIDAR y DEFENDER (≈ 3,5 min)

### 5.1 ¿Ha conservado la lectura el significado?

In [ ]:
es_num = {c: pd.api.types.is_numeric_dtype(df_leido[c]) for c in medidas}
todas_num = all(es_num.values())

comprobar(df_leido.shape == df_crudo.shape, f"Mismas filas y columnas que la carga simple {df_leido.shape}")
comprobar(todas_num, "Las seis medidas son numéricas")
comprobar(len(decisiones["columnas_numericas"]) > 0 and all(pd.api.types.is_numeric_dtype(df_leido[c]) for c in decisiones["columnas_numericas"]),
          "Las columnas que declaraste numéricas lo son")
comprobar(pd.api.types.is_datetime64_any_dtype(df_leido["Month"]), "Month es una fecha")
comprobar(df_leido[medidas].isna().sum().equals((df_crudo[medidas] == "..").sum()),
          "Nº de ausentes por columna = nº de '..' del texto original (no se pierde ni se inventa ninguna celda)")
if todas_num:
    comprobar(((df_leido[medidas] == 0).sum() == (num == 0).sum()).all(), "Los ceros siguen siendo ceros (ningún 0 real se ha vuelto ausente)")
else:
    print("✘ (pendiente) Los ceros siguen siendo ceros: necesita medidas numéricas")
if mi_clave:
    comprobar(not df_leido.duplicated(mi_clave).any(), "La clave que declaraste sigue siendo única")
comprobar(decisiones["year_es_calendario"] is not None, "Has decidido si Year es el año natural")

TOL = 1e-6
if todas_num:
    dif = (df_leido.groupby("Year")[medidas].sum() - totales).abs().max().max()
    comprobar(dif < TOL, f"Totales publicados = suma de la tabla por Year (diferencia máx. {dif:.2e}; tolerancia {TOL})")
else:
    print("✘ (pendiente) Reconciliación con los totales: necesita medidas numéricas")

### 5.2 Dos lecturas posibles de `..`

Tu compañero propone otra: sustituir `..` por 0 «para poder sumar». Aquí tienes las dos lecturas ya escritas para compararlas con **la misma** reconciliación.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, si el resultado es válido y qué significa)
df_A = pd.read_csv(RUTA_CSV, na_values=[".."], parse_dates=["Month"])   # A: '..' → ausente (NaN)
df_B = df_A.copy()
df_B[medidas] = df_B[medidas].fillna(0)                                  # B: '..' → 0

for nombre, d in [("A (NaN)", df_A), ("B (0)", df_B)]:
    dif = (d.groupby("Year")[medidas].sum() - totales).abs().max().max()
    print(f"Lectura {nombre}: diferencia máxima con los totales publicados = {dif:.2e}")

comparacion = pd.DataFrame({
    "A (NaN)": [df_A["Passengers In"].count(), df_A["Passengers In"].sum(), df_A["Passengers In"].mean()],
    "B (0)":   [df_B["Passengers In"].count(), df_B["Passengers In"].sum(), df_B["Passengers In"].mean()],
}, index=["filas con valor", "suma", "media por fila"])
print()
print("Passengers In:")
print(comparacion.round(1))

> **Pregunta silenciosa.** Las dos lecturas reconcilian con los totales y ninguna da error. ¿Qué has demostrado con esa reconciliación y qué **no**? ¿Para qué preguntas (una suma, una media por fila) importa la diferencia entre A y B?

### 5.3 Defiende tu lectura

Responde en una o dos frases cada punto. Distingue lo que demuestran los datos, lo que demuestra la documentación y lo que queda sin demostrar.

In [ ]:
mi_defensa = {
    "que_demuestran_los_datos": "",
    "que_demuestra_la_documentacion": "",
    "que_queda_sin_demostrar": "",
    "lectura_que_defiendo_y_por_que": "",
}

In [ ]:
print("Estado de tu entrega:")
print(" · grano y clave declarados:", bool(mi_grano) and bool(mi_clave))
print(" · afirmaciones en el contrato:", len(mi_contrato), "| con veredicto:", sum(1 for c in mi_contrato if c in veredictos))
print(" · decisiones de lectura rellenas:", all(v not in ([], "", None) for v in decisiones.values()))
print(" · defensa rellena:", all(bool(v) for v in mi_defensa.values()))

---
# ⏸ PARADA DEL NÚCLEO DE CLASE

**Deja de escribir.** Lo que queda (parte 6) es para casa. Ahora toca la **puesta en común** (≈ 5 min). Ten a mano:

1. Cuántas afirmaciones de tu contrato han sido PASS, FAIL y UNKNOWN.
2. Una afirmación en la que **el código corrió sin error y aun así la tabla no era lo que parecía**.
3. Una afirmación que **ningún código** puede resolver con la evidencia que tenemos: ¿qué evidencia la resolvería?

---
## Parte 6 · Para casa (≤ 20 min): completa y defiende el contrato

**Qué no se hace aquí:** nada nuevo en cuanto a conceptos. Es terminar lo mecánico y dejar por escrito lo que ya has razonado en clase.

### 6.1 Amplía tu contrato
Añade afirmaciones del menú hasta llegar a **al menos 6** (y todas las que quieras hasta las 12). Para cada una nueva: declara la predicción **antes** de ejecutar su comprobación (en `mi_contrato`) y escribe después su veredicto (en `veredictos`). Vuelve a ejecutar la tabla de la parte 3.2.

### 6.2 Justifica cada UNKNOWN
Cada UNKNOWN debe tener rellenado `que_lo_resolveria`. La celda de abajo te avisa de los que faltan.

In [ ]:
unknown = contrato[contrato["veredicto"] == "UNKNOWN"]
print("Tienes", len(unknown), "afirmación(es) UNKNOWN.")
for _, f in unknown.iterrows():
    comprobar(bool(f["que_lo_resolveria"].strip()), f"{f['id']}: has dicho qué evidencia lo resolvería")
comprobar(len(unknown) >= 1, "Tu contrato incluye al menos un UNKNOWN justificado")
comprobar(len(contrato) >= 4, f"Tu contrato tiene {len(contrato)} afirmaciones (mínimo 4)")

### 6.3 ¿Qué cambia con tu lectura?
Compara tu contrato antes y después de leer el fichero con tus decisiones. Para cada afirmación de tu contrato anota si su veredicto **cambia** con `df_leido`, y por qué sí o por qué no.

In [ ]:
tras_lectura = {
    # "C?": {"veredicto_tras_lectura": "PASS | FAIL | UNKNOWN", "porque": "una frase"},
}

### 6.4 Tu entregable
La tabla final de tu contrato: afirmación · comprobación · resultado · evidencia (≥ 4 afirmaciones, ≥ 1 UNKNOWN justificado).

In [ ]:
contrato_final = contrato.copy()
contrato_final["tras_lectura"] = [tras_lectura.get(c, {}).get("veredicto_tras_lectura", "") for c in contrato_final["id"]]
contrato_final["cambia_con_la_lectura"] = [bool(t) and t != v for t, v in zip(contrato_final["tras_lectura"], contrato_final["veredicto"])]
contrato_final[["id", "afirmacion", "prediccion", "veredicto", "evidencia", "que_lo_resolveria", "evidencia_externa", "tras_lectura"]]

### Lista de verificación antes de entregar
- [ ] Declaré mis predicciones **antes** de ejecutar las comprobaciones.
- [ ] Cada veredicto tiene una evidencia que he obtenido yo en esta tabla (una cifra o un hecho).
- [ ] Cada UNKNOWN dice **qué evidencia lo resolvería**.
- [ ] Si he usado evidencia externa (documentación, búsqueda, IA), la he **identificado** y he separado lo que establece el paquete de lo que añade esa evidencia.
- [ ] Mis decisiones de lectura están escritas y justificadas.
- [ ] Puedo explicar con mis palabras la diferencia entre «el código corre» y «la ingesta es correcta».
- [ ] Si he usado IA, entiendo cada línea que he copiado y la he comprobado sobre la tabla.

### Para quien quiera más (opcional)
- ¿Qué combinaciones de `..` aparecen entre las seis medidas? ¿Siempre las seis juntas? Ejecuta `(df_crudo[medidas] == "..").value_counts()` y describe lo que ves, **sin** concluir lo que no puedas demostrar.
- Convierte tu contrato en comprobaciones que **detengan** la lectura si algo cambia en la próxima edición del fichero (con `assert`).